# 02 · Flash Attention（IO 感知的精确注意力）

> **目标**：理解 FlashAttention 的 Tiling 算法与 Online Softmax，通过实验验证其速度和显存优势。

**目录**
1. [问题回顾](#1-问题回顾)
2. [Online Softmax](#2-Online-Softmax)
3. [Tiling 算法](#3-Tiling-算法)
4. [IO 复杂度分析](#4-IO-复杂度分析)
5. [PyTorch 模拟实现](#5-PyTorch-模拟实现)
6. [PyTorch 2.x 内置 SDPA](#6-PyTorch-2x-内置-SDPA)
7. [基准测试](#7-基准测试)
8. [反向传播简述](#8-反向传播简述)
9. [小结](#9-小结)

---

## 1 问题回顾

在 `01_standard_attention.ipynb` 中，我们得出两个关键结论：

| 指标 | 标准注意力 |
|------|------------|
| 时间复杂度 | $O(N^2 d_k)$ |
| **空间复杂度** | $\mathbf{O(N^2)}$（注意力矩阵 $A$ 须显式存储） |
| HBM 访问主项 | $O(N^2)$（读写 $S$ 和 $A$ 各两次） |

随着序列长度 $N$ 增大，**内存带宽**（HBM ↔ SRAM 的数据搬运）成为瓶颈，而非算力。

```mermaid
flowchart LR
    subgraph 标准注意力
        direction TB
        A1["读 Q,K 从 HBM"] --> B1["计算 S=QKᵀ/√dₖ"]
        B1 --> C1["写 S 到 HBM"]
        C1 --> D1["读 S，计算 softmax"]
        D1 --> E1["写 A 到 HBM"]
        E1 --> F1["读 A,V，计算 AV"]
        F1 --> G1["写 O 到 HBM"]
    end
    subgraph FlashAttention
        direction TB
        A2["按块读 Q_i, K_j, V_j"] --> B2["在 SRAM 内完成\nS/softmax/AV 累积"]
        B2 --> C2["写 O_i 到 HBM"]
    end
    标准注意力 -->|"HBM 访问 O(N²)"| 瓶颈
    FlashAttention -->|"HBM 访问 O(N²d/M)"| 优化
```

**FlashAttention 的核心洞察**：$S$ 和 $A$ 不需要被写回 HBM——只要能以分块（tiling）方式在 SRAM 内完成计算并将结果累积到输出，就可以将 HBM 访问量从 $O(N^2)$ 降到 $O(N^2 d / M)$，其中 $M$ 是 SRAM 容量。

## 2 Online Softmax

Tiling 要求我们对 **行** 做 softmax，但每块只能看到该行的一部分元素。这需要一个可以**增量更新**的 softmax 算法。

### 2.1 数值稳定的 Softmax

标准实现先减去行最大值以避免溢出：
$$
\text{softmax}(x)_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \quad m = \max_j x_j
$$

### 2.2 两状态递推（Online Softmax）

设已处理前 $t$ 个元素，维护两个累积量：
- $m^{(t)} = \max_{j \le t} x_j$（当前行最大值）
- $\ell^{(t)} = \sum_{j \le t} e^{x_j - m^{(t)}}$（当前归一化分母）

当读入第 $t+1$ 个元素 $x_{t+1}$ 时：
$$
m^{(t+1)} = \max(m^{(t)},\; x_{t+1})
$$
$$
\ell^{(t+1)} = e^{m^{(t)} - m^{(t+1)}} \cdot \ell^{(t)} + e^{x_{t+1} - m^{(t+1)}}
$$

对输出累积量 $O$ 的更新（带 $V$ 加权）：
$$
O^{(t+1)} = \frac{\ell^{(t)} \cdot e^{m^{(t)} - m^{(t+1)}}}{\ell^{(t+1)}} O^{(t)} + \frac{e^{x_{t+1} - m^{(t+1)}}}{\ell^{(t+1)}} v_{t+1}
$$

这样只需 **一趟扫描**，无需先知道全局最大值。

In [ ]:
import torch
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json, os

matplotlib.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
os.makedirs("../results", exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

In [ ]:
def online_softmax_demo(x: torch.Tensor) -> torch.Tensor:
    """
    逐元素增量计算 softmax，演示 Online Softmax 算法。
    x : (N,) 一维向量
    返回 softmax(x)
    """
    N = x.shape[0]
    m = float("-inf")   # 行最大值累积量
    l = 0.0             # 归一化分母累积量
    o = torch.zeros(N, dtype=x.dtype)  # 输出（此处演示不带 V 的版本）

    unnorm = torch.zeros(N, dtype=x.dtype)  # 存储未归一化的 exp 值

    for i in range(N):
        xi = x[i].item()
        m_new = max(m, xi)
        l_new = math.exp(m - m_new) * l + math.exp(xi - m_new)
        # 修正已累积的未归一化值
        unnorm[:i] *= math.exp(m - m_new)
        unnorm[i] = math.exp(xi - m_new)
        m, l = m_new, l_new

    return unnorm / l


# 验证与 torch.softmax 结果一致
torch.manual_seed(42)
x_test = torch.randn(16)

ref   = torch.softmax(x_test, dim=0)
ours  = online_softmax_demo(x_test)

max_err = (ref - ours).abs().max().item()
print(f"Online Softmax 最大误差 vs torch.softmax: {max_err:.2e}")
assert max_err < 1e-5
print("✓ Online Softmax 验证通过")

## 3 Tiling 算法

### 3.1 核心思想

将 $Q \in \mathbb{R}^{N \times d}$ 按行分成 $T_r$ 块，每块 $B_r$ 行；  
将 $K, V \in \mathbb{R}^{N \times d}$ 按行分成 $T_c$ 块，每块 $B_c$ 行。

外循环遍历 $K/V$ 的列块，内循环遍历 $Q$ 的行块（论文原文是外循环 $Q$，内循环 $K/V$；两种顺序等价）。每个 $(i, j)$ 块组合在 SRAM 内完成：
1. 计算局部 logits $S_{ij} = Q_i K_j^\top / \sqrt{d}$
2. 用 Online Softmax 更新 $m_i$、$\ell_i$
3. 累积输出 $O_i$

### 3.2 前向伪代码（Algorithm 1，论文 §3）

```
输入：Q, K, V ∈ R^{N×d}（存于 HBM）
参数：块大小 B_r, B_c

1. 初始化 O = 0^{N×d}, l = 0^N, m = -∞^N（存于 HBM）

2. 将 K, V 分成 T_c = ⌈N/B_c⌉ 块：K_1…K_{T_c}, V_1…V_{T_c}
   将 Q  分成 T_r = ⌈N/B_r⌉ 块：Q_1…Q_{T_r}
   同样分割 O, l, m

3. for j = 1 to T_c:              # 外循环：K/V 列块
       从 HBM 加载 K_j, V_j 到 SRAM
       for i = 1 to T_r:          # 内循环：Q 行块
           从 HBM 加载 Q_i, O_i, l_i, m_i 到 SRAM
           
           S_ij = Q_i @ K_j^T / sqrt(d)          # (B_r, B_c)
           m̃_ij = rowmax(S_ij)                   # (B_r,)
           P̃_ij = exp(S_ij - m̃_ij)              # (B_r, B_c)
           ℓ̃_ij = rowsum(P̃_ij)                  # (B_r,)
           
           m_i_new = max(m_i, m̃_ij)              # 更新全局最大值
           l_i_new = exp(m_i - m_i_new) * l_i
                   + exp(m̃_ij - m_i_new) * ℓ̃_ij
           
           O_i = diag(l_i_new)^{-1} * (
                   diag(l_i) * exp(m_i - m_i_new) * O_i
                 + exp(m̃_ij - m_i_new) * P̃_ij @ V_j
                 )                                 # 累积输出
           
           l_i, m_i = l_i_new, m_i_new
           将 O_i, l_i, m_i 写回 HBM

4. 返回 O
```

> 关键点：整个过程 **不写 $S$ 或 $A$ 到 HBM**，中间矩阵全部在 SRAM 内消费。

## 4 IO 复杂度分析

设 SRAM 大小为 $M$（元素数），块大小 $B_c \approx B_r \approx M / (4d)$（需装下 $Q_i, K_j, V_j, O_i$ 四个块）。

| 方法 | HBM 读写量（元素数） |
|------|----------------------|
| 标准注意力 | $\Theta(N d + N^2)$ |
| **FlashAttention** | $\Theta\!\left(\dfrac{N^2 d}{M}\right)$ |

当 $M \gg d$（通常 SRAM $\gg$ 单个头维度），FlashAttention 的 HBM 访问量比标准实现少 $\Theta(M/d)$ 倍——**数量级的减少**。

**直觉**：标准实现每写一次 $N^2$ 大矩阵；FlashAttention 仅写 $T_r \times T_c$ 次小块，且块内计算在 SRAM 完成，不触碰 HBM。

In [ ]:
# IO 复杂度对比可视化
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
d_k = 64
M_sram = 192 * 1024  # ~192K 元素（模拟 A100 SRAM ~192KB / float32 = 48K float32，简化）
bytes_per = 4  # float32

std_hbm   = [(N * d_k + N**2) * bytes_per / 1e6 for N in seq_lens]       # MB
flash_hbm = [(N**2 * d_k / M_sram) * bytes_per / 1e6 for N in seq_lens]  # MB

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(seq_lens, std_hbm,   marker="s", color="tomato",    label="Standard Attention")
ax.semilogy(seq_lens, flash_hbm, marker="o", color="steelblue", label="FlashAttention (theory)")

ax.set_xlabel("Sequence Length N")
ax.set_ylabel("HBM Access (MB, log scale)")
ax.set_title(f"IO Complexity: Standard vs FlashAttention  (d={d_k}, M≈{M_sram//1024}K)")
ax.set_xticks(seq_lens)
ax.set_xticklabels([str(n) for n in seq_lens], rotation=30)
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("../results/02_io_complexity.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/02_io_complexity.png")

## 5 PyTorch 模拟实现

用纯 Python + PyTorch 张量运算复现 Tiling 前向逻辑（不涉及 CUDA kernel，仅验证算法正确性）。

In [ ]:
def flash_attention_simulate(
    Q: torch.Tensor,   # (B, N, d)
    K: torch.Tensor,   # (B, N, d)
    V: torch.Tensor,   # (B, N, d)
    block_size: int = 64,
    causal: bool = False,
) -> torch.Tensor:
    """
    FlashAttention 前向 Tiling 算法的 PyTorch 模拟实现。

    注意：此实现在逻辑上等价于 FlashAttention，但并不节省显存
    （PyTorch 的 autograd 仍会保留中间变量），仅用于算法正确性验证。

    参数
    ----
    Q, K, V    : (B, N, d)  — 已经过线性投影的单头 Q/K/V
    block_size : 分块大小 B_r = B_c
    causal     : 是否应用因果掩码

    返回
    ----
    O : (B, N, d)
    """
    B, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    Br = Bc = block_size
    Tr = math.ceil(N / Br)
    Tc = math.ceil(N / Bc)

    # 输出及统计量，全部在同一设备上
    O = torch.zeros_like(Q)          # (B, N, d)
    l = torch.zeros(B, N, device=Q.device, dtype=Q.dtype)   # 归一化分母
    m = torch.full((B, N), float("-inf"), device=Q.device, dtype=Q.dtype)  # 行最大值

    for j in range(Tc):              # 外循环：K/V 列块
        j_start, j_end = j * Bc, min((j + 1) * Bc, N)
        Kj = K[:, j_start:j_end, :]  # (B, Bc', d)
        Vj = V[:, j_start:j_end, :]  # (B, Bc', d)

        for i in range(Tr):          # 内循环：Q 行块
            i_start, i_end = i * Br, min((i + 1) * Br, N)
            Qi = Q[:, i_start:i_end, :]  # (B, Br', d)
            Oi = O[:, i_start:i_end, :]  # (B, Br', d)  — view
            li = l[:, i_start:i_end]     # (B, Br')
            mi = m[:, i_start:i_end]     # (B, Br')

            # 局部 logits：(B, Br', Bc')
            Sij = torch.bmm(Qi, Kj.transpose(-2, -1)) * scale

            # 因果掩码：只允许关注 j <= i 的位置
            if causal:
                # query 位置 [i_start, i_end)，key 位置 [j_start, j_end)
                q_idx = torch.arange(i_start, i_end, device=Q.device).unsqueeze(1)  # (Br',1)
                k_idx = torch.arange(j_start, j_end, device=Q.device).unsqueeze(0)  # (1,Bc')
                mask = k_idx > q_idx   # True → 遮蔽
                Sij = Sij.masked_fill(mask.unsqueeze(0), float("-inf"))

            # Online Softmax 更新
            m_tilde = Sij.max(dim=-1).values          # (B, Br')  局部最大
            P_tilde = torch.exp(Sij - m_tilde.unsqueeze(-1))  # (B, Br', Bc')
            l_tilde = P_tilde.sum(dim=-1)             # (B, Br')  局部 sum

            mi_new = torch.maximum(mi, m_tilde)       # 全局最大
            li_new = (
                torch.exp(mi - mi_new) * li
                + torch.exp(m_tilde - mi_new) * l_tilde
            )                                         # 全局 sum

            # 累积输出
            # O_i_new = diag(1/l_new) * (diag(l_old)*exp(m_old-m_new)*O_i + exp(m_tilde-m_new)*P_tilde@V_j)
            alpha = (torch.exp(mi - mi_new) * li / li_new).unsqueeze(-1)       # (B, Br', 1)
            beta  = (torch.exp(m_tilde - mi_new) / li_new).unsqueeze(-1)       # (B, Br', 1)
            Oi_new = alpha * Oi + beta * torch.bmm(P_tilde, Vj)  # (B, Br', d)

            # 写回
            O[:, i_start:i_end, :] = Oi_new
            l[:, i_start:i_end]    = li_new
            m[:, i_start:i_end]    = mi_new

    return O

In [ ]:
# ---- 正确性验证 ----
torch.manual_seed(0)
B, N, d = 2, 128, 64
Q = torch.randn(B, N, d, device=device)
K = torch.randn(B, N, d, device=device)
V = torch.randn(B, N, d, device=device)

# 参考：朴素注意力（来自 01_standard_attention.ipynb 中的实现）
def naive_attention_ref(Q, K, V):
    scale = 1.0 / math.sqrt(Q.size(-1))
    scores = torch.bmm(Q, K.transpose(-2, -1)) * scale
    attn = torch.softmax(scores, dim=-1)
    return torch.bmm(attn, V)

ref_out  = naive_attention_ref(Q, K, V)

# 不同块大小测试
print(f"{'块大小':>8}  {'最大绝对误差':>14}  {'通过':>5}")
print("-" * 35)
for bs in [16, 32, 64, 128]:
    out = flash_attention_simulate(Q, K, V, block_size=bs)
    err = (out - ref_out).abs().max().item()
    ok  = "✓" if err < 1e-4 else "✗"
    print(f"{bs:>8}  {err:>14.2e}  {ok:>5}")

In [ ]:
# ---- 因果掩码验证 ----
torch.manual_seed(1)
B, N, d = 1, 32, 16
Q2 = torch.randn(B, N, d)
K2 = torch.randn(B, N, d)
V2 = torch.randn(B, N, d)

# 朴素因果注意力
scale = 1.0 / math.sqrt(d)
S = torch.bmm(Q2, K2.transpose(-2, -1)) * scale
causal_mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
S_masked = S.masked_fill(causal_mask.unsqueeze(0), float("-inf"))
ref_causal = torch.bmm(torch.softmax(S_masked, dim=-1), V2)

flash_causal = flash_attention_simulate(Q2, K2, V2, block_size=8, causal=True)
err_causal = (flash_causal - ref_causal).abs().max().item()
print(f"因果掩码最大误差: {err_causal:.2e}")
assert err_causal < 1e-4
print("✓ 因果掩码验证通过")

## 6 PyTorch 2.x 内置 SDPA

PyTorch 2.0+ 在 `torch.nn.functional.scaled_dot_product_attention` 中内置了多个 backend，会根据硬件和输入形状自动选择：

| Backend | 触发条件 | 说明 |
|---------|----------|------|
| `FLASH_ATTENTION` | CUDA + fp16/bf16 + 无自定义掩码 | 调用 FlashAttention CUDA kernel |
| `EFFICIENT_ATTENTION` | CUDA + 其他条件 | xFormers memory-efficient attention |
| `MATH`  | 兜底 | 等价于标准朴素实现 |

可通过 `torch.backends.cuda.sdp_kernel` 上下文管理器强制指定 backend。

In [ ]:
torch.manual_seed(42)
B, H, N, d = 2, 8, 256, 64

# F.scaled_dot_product_attention 期望 (B, H, N, d)
Qh = torch.randn(B, H, N, d, device=device)
Kh = torch.randn(B, H, N, d, device=device)
Vh = torch.randn(B, H, N, d, device=device)

# 默认（自动选择 backend）
out_default = F.scaled_dot_product_attention(Qh, Kh, Vh)
print(f"默认 backend 输出形状: {out_default.shape}")

# 可用 backend 检查
if torch.cuda.is_available():
    # 强制使用 math backend（等价于朴素实现），用于对比
    with torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    ):
        out_math = F.scaled_dot_product_attention(Qh, Kh, Vh)

    diff = (out_default - out_math).abs().max().item()
    print(f"默认 vs math backend 最大差异: {diff:.2e}")
    print("（差异由不同 kernel 的浮点计算顺序导致，属正常现象）")
else:
    print("CPU 模式：仅 math backend 可用")
    print("完整 Flash Attention backend 需要 CUDA GPU。")

# 因果掩码（is_causal=True）
out_causal = F.scaled_dot_product_attention(Qh, Kh, Vh, is_causal=True)
print(f"因果注意力输出形状: {out_causal.shape}")

In [ ]:
# 打印当前 PyTorch SDPA 支持的 backend 状态
if torch.cuda.is_available():
    print("SDPA backend 可用性：")
    for name, fn in [
        ("Flash Attention",    lambda: torch.backends.cuda.flash_sdp_enabled()),
        ("Mem-Efficient Attn", lambda: torch.backends.cuda.mem_efficient_sdp_enabled()),
        ("Math (fallback)",    lambda: torch.backends.cuda.math_sdp_enabled()),
    ]:
        try:
            status = fn()
        except AttributeError:
            status = "N/A"
        print(f"  {name:25s}: {status}")
else:
    print("CPU 模式，跳过 CUDA backend 状态检查。")

## 7 基准测试

对比三种实现在不同序列长度下的**延迟**和**显存峰值**：
1. `naive_attention`（手写朴素实现，来自 Notebook 01）
2. `flash_attention_simulate`（本 Notebook 的 Tiling 模拟，CPU/GPU 均可）
3. `F.scaled_dot_product_attention`（PyTorch 内置，GPU 上自动启用 Flash Attention kernel）

In [ ]:
import torch.utils.benchmark as benchmark

def measure(fn, *args, n_warmup=3, n_repeat=10, label=""):
    dev = args[0].device
    for _ in range(n_warmup):
        fn(*args)
    if dev.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(dev)
        fn(*args)
        torch.cuda.synchronize()
        mem_mb = torch.cuda.max_memory_allocated(dev) / 1e6
    else:
        mem_mb = float("nan")

    t = benchmark.Timer(
        stmt="fn(*args)",
        globals={"fn": fn, "args": args},
        label=label, num_threads=1,
    )
    r = t.timeit(n_repeat)
    return {"label": label, "latency_ms": r.mean * 1e3,
            "memory_mb": mem_mb, "seq_len": args[0].shape[1]}


# --- 包装函数 ---
def run_naive(Q, K, V):
    return naive_attention_ref(Q, K, V)

def run_flash_sim(Q, K, V):
    return flash_attention_simulate(Q, K, V, block_size=64)

def run_sdpa(Q, K, V):
    # F.sdpa 需要 (B,H,N,d)；这里 H=1
    return F.scaled_dot_product_attention(
        Q.unsqueeze(1), K.unsqueeze(1), V.unsqueeze(1)
    ).squeeze(1)


BENCH_SEQ  = [128, 256, 512, 1024]
if device == "cuda":
    BENCH_SEQ += [2048, 4096]
D_K, BATCH = 64, 1

records = []
impls = [
    ("naive",         run_naive),
    ("flash_sim",     run_flash_sim),
    ("F.sdpa",        run_sdpa),
]

print(f"{'N':>6}  {'实现':>12}  {'延迟(ms)':>10}  {'显存(MB)':>10}")
print("-" * 48)

for N in BENCH_SEQ:
    torch.manual_seed(0)
    Q_ = torch.randn(BATCH, N, D_K, device=device)
    K_ = torch.randn(BATCH, N, D_K, device=device)
    V_ = torch.randn(BATCH, N, D_K, device=device)

    for name, fn in impls:
        try:
            rec = measure(fn, Q_, K_, V_, label=name)
            rec["impl"] = name
            records.append(rec)
            print(f"{N:>6}  {name:>12}  {rec['latency_ms']:>10.3f}  {rec['memory_mb']:>10.1f}")
        except Exception as e:
            print(f"{N:>6}  {name:>12}  ERROR: {e}")

with open("../results/02_benchmark_flash_attention.json", "w") as f:
    json.dump(records, f, indent=2)
print("\n结果已保存至 results/02_benchmark_flash_attention.json")

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {"naive": "tomato", "flash_sim": "orange", "F.sdpa": "steelblue"}

for ax, metric, ylabel, title in [
    (axes[0], "latency_ms", "Latency (ms)",    "Attention Latency vs Sequence Length"),
    (axes[1], "memory_mb",  "Peak Memory (MB)", "Attention Memory vs Sequence Length"),
]:
    for impl, grp in df.groupby("impl"):
        g = grp.sort_values("seq_len")
        ax.plot(g["seq_len"], g[metric],
                marker="o", label=impl, color=colors.get(impl, None))
    ax.set_xlabel("Sequence Length N")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(f"Flash Attention Benchmark  |  d={D_K}, batch={BATCH}, device={device}")
plt.tight_layout()
plt.savefig("../results/02_benchmark_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/02_benchmark_plot.png")

In [ ]:
# --- 加速比分析：F.sdpa vs naive ---
pivot = df.pivot_table(index="seq_len", columns="impl", values="latency_ms")
if "naive" in pivot.columns and "F.sdpa" in pivot.columns:
    pivot["speedup_sdpa_vs_naive"] = pivot["naive"] / pivot["F.sdpa"]
    print("\nF.sdpa 相对于 naive 的加速比：")
    print(pivot[["naive", "F.sdpa", "speedup_sdpa_vs_naive"]].to_string(float_format="{:.3f}".format))
else:
    print("\n（部分 impl 基准数据缺失，跳过加速比计算）")

## 8 反向传播简述

标准实现的反向传播需要保存 $A \in \mathbb{R}^{N \times N}$ 用于计算梯度，这与前向同样消耗 $O(N^2)$ 显存。

FlashAttention 的反向传播采用**重计算（Recomputation）**策略：

```mermaid
flowchart LR
    subgraph 前向
        F1["读 Q,K,V"] --> F2["Tiling 计算"]
        F2 --> F3["写 O, l, m 到 HBM\n（不写 S 或 A）"]
    end
    subgraph 反向
        B1["读 Q,K,V,O,l,m,dO"] --> B2["重新计算 S,A\n（在 SRAM 内）"]
        B2 --> B3["计算 dQ, dK, dV"]
        B3 --> B4["写 dQ, dK, dV 到 HBM"]
    end
    前向 -->|"保存 O,l,m\n（O(N) 额外存储）"| 反向
```

| | 前向 | 反向 |
|--|------|------|
| 保存到 HBM | $O, \ell, m$（共 $O(Nd)$） | 无额外存储 |
| 反向所需中间量 | — | 重计算 $S_{ij}, A_{ij}$（SRAM 内） |
| **总显存** | $O(Nd)$ | vs 标准 $O(N^2)$ |

代价是反向传播需要重新读一次 $Q, K, V$，但由于 HBM 访问量仍远小于标准实现写 $A$ 的开销，整体仍更快。

### FlashAttention v2 的改进

| 点 | v1 | v2 |
|----|----|----|
| 外循环 | $K/V$ 块 | $Q$ 块（更利于 GPU warp 并行） |
| FLOPs（非因果） | $4 N^2 d$ | $\approx 2 N^2 d$ |
| 因果掩码优化 | 无 | 跳过全遮蔽块，约 2× 加速 |
| 序列并行 | 不支持 | 支持（跨 GPU 分割序列维度） |

In [ ]:
# 验证 flash_attention_simulate 支持梯度（autograd 可追踪）
torch.manual_seed(7)
B, N, d = 1, 32, 16
Qg = torch.randn(B, N, d, requires_grad=True)
Kg = torch.randn(B, N, d, requires_grad=True)
Vg = torch.randn(B, N, d, requires_grad=True)

out_flash = flash_attention_simulate(Qg, Kg, Vg, block_size=8)
loss = out_flash.sum()
loss.backward()

print("梯度形状：")
print(f"  dQ : {Qg.grad.shape}")
print(f"  dK : {Kg.grad.shape}")
print(f"  dV : {Vg.grad.shape}")

# 对比朴素实现的梯度
Qg2 = Qg.detach().clone().requires_grad_(True)
Kg2 = Kg.detach().clone().requires_grad_(True)
Vg2 = Vg.detach().clone().requires_grad_(True)
out_naive = naive_attention_ref(Qg2, Kg2, Vg2)
out_naive.sum().backward()

for name, g1, g2 in [("dQ", Qg.grad, Qg2.grad),
                      ("dK", Kg.grad, Kg2.grad),
                      ("dV", Vg.grad, Vg2.grad)]:
    err = (g1 - g2).abs().max().item()
    print(f"  {name} 梯度最大误差: {err:.2e}  {'✓' if err < 1e-4 else '✗'}")

## 9 小结

| 结论 | 细节 |
|------|------|
| **算法等价性** | FlashAttention 与标准注意力数学结果完全一致，是**精确**而非近似算法 |
| **显存节省** | 通过不物化 $N\times N$ 矩阵，将显存从 $O(N^2)$ 降至 $O(N)$ |
| **速度提升** | HBM 访问量从 $O(N^2)$ 降至 $O(N^2 d/M)$，GPU 上实测 2–4× 加速 |
| **因果掩码零开销** | Tiling 天然支持块级掩码，不引入额外计算 |
| **PyTorch 集成** | `F.scaled_dot_product_attention` 在 GPU + fp16/bf16 时自动调用 Flash Attention kernel |
| **反向传播** | 通过重计算 $S, A$，反向也保持 $O(N)$ 额外显存 |

### 下一步：量化

FlashAttention 解决了**长序列**的显存和带宽问题。另一个正交的优化方向是**量化**：将权重和激活值从 FP32/FP16 压缩到 INT8/INT4，在精度可接受的前提下进一步减少显存和计算量。详见 **`03_quantization.ipynb`**。

---
*参考文献*
- Dao et al. \"FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness\", NeurIPS 2022.
- Dao et al. \"FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning\", ICLR 2024.
- Milakov & Gimelshein. \"Online normalizer calculation for softmax\", arXiv 2018.